In [ ]:
from datetime import date

import numpy as np
import pandas as pd
import yaml
from sqlalchemy import create_engine


# database connections 

In [ ]:
with open('../config.yml', 'r') as f:
    config = yaml.safe_load(f)
    config_co = config['CO_SA']
    config_etl = config['ETL_PRO']

# Construct the database URL
url_co = (f"{config_co['drivername']}://{config_co['user']}:{config_co['password']}@{config_co['host']}:"
          f"{config_co['port']}/{config_co['dbname']}")
url_etl = (f"{config_etl['drivername']}://{config_etl['user']}:{config_etl['password']}@{config_etl['host']}:"
           f"{config_etl['port']}/{config_etl['dbname']}")
# Create the SQLAlchemy Engine
co_sa = create_engine(url_co)
etl_conn = create_engine(url_etl)

# Extract

`sede` no trae el nombre del departamento directamente: hay que hacer join `sede -> ciudad -> departamento`.

In [ ]:
dim_sede = pd.read_sql_query('''
    SELECT
        s.sede_id,
        s.nombre          AS nombre_sede,
        s.direccion,
        s.telefono,
        s.nombre_contacto,
        c.nombre          AS ciudad,
        d.nombre          AS departamento,
        s.cliente_id
    FROM sede s
    LEFT JOIN ciudad c ON s.ciudad_id = c.ciudad_id
    LEFT JOIN departamento d ON c.departamento_id = d.departamento_id
''', co_sa)

dim_sede.info()

In [ ]:
dim_sede.head()

# Transformations

In [ ]:
# mismo patron de limpieza de nulos/vacios usado en dim_estado, dim_mensajero y dim_tipo_servicio
dim_sede.replace({np.nan: 'no aplica', ' ': 'no aplica', '': 'no_aplica'}, inplace=True)
dim_sede['saved'] = date.today()

In [ ]:
dim_sede.describe(include='all')

# load

In [ ]:
# NOTA: cambiado de if_exists='replace' a 'append' porque la tabla ahora se crea por DDL
# (ver sqlscripts.yml) con su PK real. 'replace' borraria esa tabla y la reemplazaria sin
# restricciones. Antes de correr este notebook, asegurate que main.py ya ejecuto el DDL
# (o ejecuta el script de sqlscripts.yml manualmente si la tabla aun no existe).
dim_sede.to_sql('dim_sede', etl_conn, if_exists='append', index=False)